In [1]:
import pandas as pd
from pprint import pprint
import json
import re
from typing import Dict, List,Any

In [2]:
ds_7dayWorkout_900 = pd.read_json("Original datasets/7dayWorkout_900.json", lines=True)
# ds_plannerStudy_628 = pd.read_json("Original datasets/plannerStudy_628.json", lines=True)
# ds_workouts_101 = pd.read_json("Original datasets/workouts_101.json", lines=True)

In [3]:
# pprint(ds_7dayWorkout_900['train'][0])
# pprint(ds_7dayWorkout_900["Context"][0])
# pprint(ds_7dayWorkout_900["Response"][0])
# print(ds_7dayWorkout_900)
samples = []
# pprint(ds_7dayWorkout_900['Context'][0]+ds_7dayWorkout_900['Response'][0])
for row in range(5):
    samples.append(
        {col: ds_7dayWorkout_900[col][row] for col in ds_7dayWorkout_900.columns}
    )
print(samples)

[{'Context': 'I am a 21-year-old male with a height of 180 cm and a weight of 55 kg. My primary goal is endurance, and I aim to reach a target weight of 45 kg. I am a beginner at the gym. Can you create a 7-day workout plan for me?', 'Response': 'Monday:\n- Russian Twists: 3 sets of 20 reps\n- Pull-ups: 3 sets of 10 reps\n- Squats: 3 sets of 10 reps\nTuesday:\n- Burpees: 3 sets of 6 reps\n- Running: 3 sets of 20 reps\n- Shoulder Press: 3 sets of 20 reps\nWednesday:\n- Rest Day\nThursday:\n- Planks: 3 sets of 6 reps\n- Tricep Dips: 3 sets of 15 reps\n- Shoulder Press: 3 sets of 8 reps\nFriday:\n- Walking: 3 sets of 10 reps\n- Planks: 3 sets of 20 reps\n- Squats: 3 sets of 12 reps\nSaturday:\n- Rest Day\nSunday:\n- Rest Day'}, {'Context': 'I am a 45-year-old male with a height of 200 cm and a weight of 59 kg. My primary goal is weight loss, and I aim to reach a target weight of 54 kg. I am a intermediate at the gym. Can you create a 7-day workout plan for me?', 'Response': 'Monday:\n- Le

In [4]:
WEEKDAYS = r"\\b(monday|tuesday|wednesday|thursday|friday|saturday|sunday)\\b"
DAY_NUMBER = r"\\bday\\s*\\d+\\b"
DAY_PLAN = r"\\b\\d+[- ]day plan\\b"


FITNESS_KEYWORDS = [
    "workout",
    "exercise",
    "gym",
    "cardio",
    "endurance",
    "run",
    "walking",
]
STUDY_KEYWORDS = ["study", "exam", "learn", "course", "revision"]
HABIT_SIGNALS = ["daily", "weekly", "routine", "habit", "every", "-day"]


SAFE_BMI_MIN = 18.5

In [ ]:
def normalize_text(text: str) -> str:
    text = text.lower()
    text = re.sub(WEEKDAYS, "", text, flags=re.IGNORECASE)
    text = re.sub(DAY_NUMBER, "", text, flags=re.IGNORECASE)
    text = re.sub(DAY_PLAN, "recurring plan", text, flags=re.IGNORECASE)
    text = re.sub(r"[\n\r\t]+", " ", text)
    text = re.sub(r"\s{2,}", " ", text)
    return text.strip()


def extract_keywords(text: str) -> List[str]:
    keywords = []
    for word in FITNESS_KEYWORDS + STUDY_KEYWORDS:
        if word in text:
            keywords.append(word)
    return list(set(keywords))


def infer_goal_type(keywords: List[str]) -> str:
    if any(k in FITNESS_KEYWORDS for k in keywords):
        return "fitness"
    if any(k in STUDY_KEYWORDS for k in keywords):
        return "study"
    return "general"


def infer_goal_category(text: str) -> str:
    for signal in HABIT_SIGNALS:
        if signal in text:
            return "habit"
    return "one_time"


def sanitize_unsafe_targets(text: str) -> (str, List[str]):
    notes = []
    weight_match = re.search(r"weight of (\\d+)", text)
    height_match = re.search(r"height of (\\d+)", text)

    if weight_match and height_match:
        weight = float(weight_match.group(1))
        height_cm = float(height_match.group(1))
        height_m = height_cm / 100
        bmi = weight / (height_m**2)
        if bmi < SAFE_BMI_MIN:
            text = re.sub(r"target weight of \\d+ kg", "improve overall fitness", text)
            notes.append("removed unsafe weight target")

    return text, notes


def extract_experience_level(text: str) -> str:
    if "beginner" in text:
        return "beginner"
    if "intermediate" in text:
        return "intermediate"
    if "advanced" in text:
        return "advanced"
    return "unspecified"

def extract_physical_constraints(text: str) -> str:
    """
    Extracts physical constraints (age, gender, height, weight, experience level)
    from a free-text description and returns them in a structured dict.
    """

    # Regex patterns
    age_gender_pattern = r"(\d{1,2})-year-old\s+(male|female)"
    height_pattern = r"height of\s+(\d{2,3})\s*cm"
    weight_pattern = r"weight of\s+(\d{2,3})\s*kg"
    experience_pattern = r"\b(beginner|intermediate|advanced)\b"

    # Extract values
    age_gender = re.search(age_gender_pattern, text.lower())
    height = re.search(height_pattern, text.lower())
    weight = re.search(weight_pattern, text.lower())
    experience = re.search(experience_pattern, text.lower())

    # Build structured string
    physical_constraints = []
    if age_gender:
        physical_constraints.append(
            f"{age_gender.group(1)}-year-old {age_gender.group(2)}"
        )
    if height:
        physical_constraints.append(f"{height.group(1)} cm")
    if weight:
        physical_constraints.append(f"{weight.group(1)} kg")
    if experience:
        physical_constraints.append(f"{experience.group(1)} experience level")

    return "; ".join(physical_constraints)

def generate_goal_name(text: str) -> str:
    """
    Extracts the primary goal and routine type from the context
    and generates a concise goal name.
    """
    # Look for primary goal
    goal_match = re.search(r"primary goal is (\w+)", text.lower())
    goal = goal_match.group(1) if goal_match else "fitness"

    # Look for routine type (7-day workout plan / fitness routine)
    routine_match = re.search(
        r"(\d+-day [\w\s]+routine|\d+-day workout plan)", text.lower()
    )
    routine = routine_match.group(1) if routine_match else "fitness routine"

    # Build goal name
    return f"Improve {goal} and complete a {routine}"


# Example usage
clean_context = (
    "Improve i am a 21-year-old male with a height of 180 cm and a weight of 55 "
    "kg. my primary goal is endurance, and i aim to reach a target weight of 45 "
    "kg. i am a beginner at the gym. can you create a 7-day workout plan for me? "
    "through a structured fitness routine"
)

# print(generate_goal_name(clean_context))

# Example usage
# text = (
#     "i am a 21-year-old male with a height of 180 cm and a weight of 55 kg. "
#     "my primary goal is endurance, and i aim to reach a target weight of 45 kg. "
#     "i am a beginner at the gym. can you create a 7-day workout plan for me?"
# )

# print(extract_physical_constraints(text))

In [6]:
time_map = {
    "squats": 6,  # seconds per rep
    "pull-ups": 8,
    "russian twists": 5,
    "burpees": 10,
    "shoulder press": 6,
    "tricep dips": 6,
    "planks": 5,  # reps = seconds
    "walking": 60,  # reps = minutes
    "running": 60,  # reps = minutes
    "cycling": 60,  # reps = minutes
    "kettlebell swings": 6,
}
def parse_tasks(task_desc: str):
    pattern = r"-\s*([a-z ]+):\s*(\d+)\s*sets\s*of\s*(\d+)\s*reps"
    matches = re.findall(pattern, task_desc.lower())
    results = []
    for exercise, sets, reps in matches:
        results.append(
            {"exercise": exercise.strip(), "sets": int(sets), "reps": int(reps)}
        )
    return results


# task_desc = "- burpees: 3 sets of 6 reps\n- running: 3 sets of 20 reps\n- shoulder press: 3 sets of 20 reps"
# print(parse_tasks(task_desc))

def estimate_duration(task_desc: str) -> int:
    total_minutes = 0
    exercises=parse_tasks(task_desc)
    for line in exercises:
        if line:
            exercise, sets, reps = line['exercise'], line['sets'], line['reps']
            sets, reps = int(sets), int(reps)
            exercise = exercise.strip().lower()
            if exercise in ["walking", "running", "cycling"]:
                # reps = minutes
                total_minutes += sets * reps
            elif exercise == "planks":
                # reps = seconds
                total_minutes += (sets * reps) / 60
            else:
                # strength moves: convert seconds → minutes
                seconds = sets * reps * time_map.get(exercise, 6)
                total_minutes += seconds / 60
            # print(
            #     f"Exercise: {exercise}, Sets: {sets}, Reps: {reps}, Total Minutes So Far: {total_minutes}"
            # )
    return round(total_minutes, 1)


# Example
# desc = "- burpees: 3 sets of 6 reps\n- running: 3 sets of 20 reps\n- shoulder press: 3 sets of 20 reps"
# print("Estimated duration:", estimate_duration(desc), "minutes")

In [ ]:
def extract_tasks(response: str) -> List[Dict[str, Any]]:
    """
    Splits the clean_response by weekdays.
    Returns a list of task objects with index as day number.
    """
    days = [
        "monday",
        "tuesday",
        "wednesday",
        "thursday",
        "friday",
        "saturday",
        "sunday",
    ]

    # Convert response to lowercase for matching
    response_lower = response.lower()

    # Find all day positions in the text
    day_positions = []
    for day in days:
        pos = response_lower.find(day)
        if pos != -1:
            day_positions.append((pos, day))

    # Sort by position
    # day_positions.sort(key=lambda x: x[0])

    # Extract tasks for each day
    day_tasks = {}
    for i, (pos, day) in enumerate(day_positions):
        start = pos + len(day)
        end = (
            day_positions[i + 1][0]
            if i + 1 < len(day_positions)
            else len(response_lower)
        )

        tasks = response_lower[start:end].strip()
        # print(day ,"---->" ,tasks)
        tasks = re.sub(r"^[\s:]+", "", tasks)
        # print ("tasks ----->", tasks,"\n=============================")

        day_tasks[day] = tasks

    # Build structured output
    results = []
    for idx, day in enumerate(days):
        task_desc = day_tasks.get(day, None)
        # ^\s*[- ]* → optional leading spaces/dashes # ([^:]+) → capture everything up to the colon
        pattern = r"^\s*[- ]*([^:]+)"
        workout_names = re.findall(pattern, task_desc, flags=re.MULTILINE)
        estimate_duration_minutes = estimate_duration(task_desc) if task_desc else 0
        min_duration =max(estimate_duration_minutes-5,5)
        max_duration = estimate_duration_minutes +10

        obj = {
            "index": idx + 1,
            "task": ", ".join(workout_names) if workout_names else "",
            "description": task_desc if task_desc else "",
            "is_repeatable": True,
            "gap_flag": True if "rest" in (task_desc or "") else False,
            "repeat_frequency": 7,
            "estimated_duration_minutes": {"min": min_duration, "max": max_duration},
        }
        results.append(obj)

    return results

In [25]:
# Test the function with the example
test_response = (
    "Monday:\n- Russian Twists: 3 sets of 20 reps\n- Pull-ups: 3 sets of 10 reps\n- Squats: 3 sets of 10 reps\nTuesday:\n- Burpees: 3 sets of 6 reps\n- Running: 3 sets of 20 reps\n- Shoulder Press: 3 sets of 20 reps\nWednesday:\n- Rest Day\nThursday:\n- Planks: 3 sets of 6 reps\n- Tricep Dips: 3 sets of 15 reps\n- Shoulder Press: 3 sets of 8 reps\nFriday:\n- Walking: 3 sets of 10 reps\n- Planks: 3 sets of 20 reps\n- Squats: 3 sets of 12 reps\nSaturday:\n- Rest Day\nSunday:\n- Rest Day"
)

result = extract_tasks(test_response)
print(json.dumps(result, indent=2))
# pprint(result)

[
  {
    "index": 1,
    "task": "russian twists, pull-ups, squats",
    "description": "- russian twists: 3 sets of 20 reps\n- pull-ups: 3 sets of 10 reps\n- squats: 3 sets of 10 reps",
    "is_repeatable": true,
    "gap_flag": false,
    "repeat_frequency": 7,
    "estimated_duration_minutes": {
      "min": 6.0,
      "max": 21.0
    }
  },
  {
    "index": 2,
    "task": "burpees, running, shoulder press",
    "description": "- burpees: 3 sets of 6 reps\n- running: 3 sets of 20 reps\n- shoulder press: 3 sets of 20 reps",
    "is_repeatable": true,
    "gap_flag": false,
    "repeat_frequency": 7,
    "estimated_duration_minutes": {
      "min": 64.0,
      "max": 79.0
    }
  },
  {
    "index": 3,
    "task": "rest day",
    "description": "- rest day",
    "is_repeatable": true,
    "gap_flag": true,
    "repeat_frequency": 7,
    "estimated_duration_minutes": {
      "min": 5,
      "max": 10
    }
  },
  {
    "index": 4,
    "task": "planks, tricep dips, shoulder press",
   

In [31]:
def stage1_preprocess(row: Dict[str, str]) -> Dict:
    raw_context = row.get("Context", "")
    raw_response = row.get("Response", "")

    clean_context = normalize_text(raw_context)
    clean_response = normalize_text(raw_response)
    tasks= extract_tasks(clean_response)
    clean_context, safety_notes = sanitize_unsafe_targets(clean_context)

    keywords = extract_keywords(clean_context + " " + clean_response)
    goal_type = infer_goal_type(keywords)
    goal_category = infer_goal_category(clean_context + " " + clean_response)
    experience_level = extract_experience_level(clean_context)
    physical_constraints=extract_physical_constraints(clean_context)
    goal_name=generate_goal_name(clean_context) 
    return {
        "goal": goal_name,
        "goal_category": goal_category,
        "goal_type": goal_type,
        "time_horizon": "long_term",
        "description": "",
        "baseline": {
            "fixed_constraints": "",
            "physical_constraints": physical_constraints,
            "non_negotiables": "",
        },
        "clean_context": clean_context,
        "goal_keywords": keywords,
        "experience_level": experience_level,
        "safety_notes": safety_notes,
        "tasks": tasks,
    }


# # Apply the preprocessing function to the entire dataset
processed_data = ds_7dayWorkout_900.apply(stage1_preprocess, axis=1)

# # Save the processed data to a new JSON file
# processed_data.to_json("stage2/processed_7dayWorkout_900.json", orient="records", lines=True)

In [32]:
pprint(processed_data[0])

{'baseline': {'fixed_constraints': '',
              'non_negotiables': '',
              'physical_constraints': '21-year-old male; 180 cm; 55 kg; '
                                      'beginner experience level'},
 'clean_context': 'i am a 21-year-old male with a height of 180 cm and a '
                  'weight of 55 kg. my primary goal is endurance, and i aim to '
                  'reach a target weight of 45 kg. i am a beginner at the gym. '
                  'can you create a 7-day workout plan for me?',
 'description': '',
 'experience_level': 'beginner',
 'goal': 'Improve endurance and complete a 7-day workout plan',
 'goal_category': 'habit',
 'goal_keywords': ['endurance', 'run', 'workout', 'gym', 'walking'],
 'goal_type': 'fitness',
 'safety_notes': [],
 'tasks': [{'description': '- russian twists: 3 sets of 20 reps - pull-ups: 3 '
                           'sets of 10 reps - squats: 3 sets of 10 reps',
            'estimated_duration_minutes': {'max': 21.0, 'min': 6.0}

In [34]:
for i in range(10):
    pprint(processed_data.iloc[i]["clean_context"])

('i am a 21-year-old male with a height of 180 cm and a weight of 55 kg. my '
 'primary goal is endurance, and i aim to reach a target weight of 45 kg. i am '
 'a beginner at the gym. can you create a 7-day workout plan for me?')
('i am a 45-year-old male with a height of 200 cm and a weight of 59 kg. my '
 'primary goal is weight loss, and i aim to reach a target weight of 54 kg. i '
 'am a intermediate at the gym. can you create a 7-day workout plan for me?')
('i am a 44-year-old female with a height of 194 cm and a weight of 55 kg. my '
 'primary goal is weight loss, and i aim to reach a target weight of 50 kg. i '
 'am a advanced at the gym. can you create a 7-day workout plan for me?')
('i am a 52-year-old female with a height of 196 cm and a weight of 73 kg. my '
 'primary goal is weight loss, and i aim to reach a target weight of 68 kg. i '
 'am a intermediate at the gym. can you create a 7-day workout plan for me?')
('i am a 31-year-old male with a height of 169 cm and a weight